##### Imports & setup

In [1]:
import pandas as pd
import numpy as np
from magcvs_library.science import extract_features, get_lightcurve_data, sort_negative
# Disabling FutureWarnings:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

## Getting data

### Positive class

In [2]:
# Reading IDs of selected magnetic Cataclysmic Variables:
all_positive_Ids = pd.read_csv('../data/magcvs_objids_all_years_p3_clean.csv')['objectId'].values.flatten()
for i, id in enumerate(all_positive_Ids):
    if len(id) > 12:
        all_positive_Ids[i] = id.replace('[', '').replace(']', '').replace("'", '')[0:12]
all_positive_Ids = np.unique(all_positive_Ids)

polars_Ids = pd.read_csv('../data/polars_objids_all_years_p3_clean.csv')['objectId'].values.flatten()
for i, id in enumerate(polars_Ids):
    if len(id) > 12:
        polars_Ids[i] = id.replace('[', '').replace(']', '').replace("'", '')[0:12]
polars_Ids = np.unique(polars_Ids)

In [3]:
# Getting corresponding lightcuves with FINK API:
all_positive_lc_g, all_positive_lc_r = get_lightcurve_data(all_positive_Ids, cut=100, time_range_split=False)
polars_lc_g, polars_lc_r = get_lightcurve_data(polars_Ids, cut=100, time_range_split=False)

100%|██████████| 94/94
100%|██████████| 50/50


In [4]:
# Saving to parquet:
all_positive_lc_g.to_parquet('../../data/magcvs/light_curves_g.parquet')
all_positive_lc_r.to_parquet('../../data/magcvs/light_curves_r.parquet')
polars_lc_g.to_parquet('../../data/polars/light_curves_g.parquet')
polars_lc_r.to_parquet('../../data/polars/light_curves_r.parquet')

### Negative class

In [5]:
# Getting pre-downloaded lightcurves from all objects in the FINK database from 2020 to 2024 (with a cut on 1-year time ranges):
negative_lc_2020 = pd.read_parquet('../../data/All_2020/')
negative_lc_2021 = pd.read_parquet('../../data/All_2021/')
negative_lc_2022 = pd.read_parquet('../../data/All_2022/')
negative_lc_2023 = pd.read_parquet('../../data/All_2023/')
negative_lc_2024 = pd.read_parquet('../../data/All_2024/')

In [6]:
# Splitting the lightcurves by filter and removing potential positive objects:
negative_lc_2020_g, negative_lc_2020_r = sort_negative(negative_lc_2020, all_positive_Ids)
negative_lc_2021_g, negative_lc_2021_r = sort_negative(negative_lc_2021, all_positive_Ids)
negative_lc_2022_g, negative_lc_2022_r = sort_negative(negative_lc_2022, all_positive_Ids)
negative_lc_2023_g, negative_lc_2023_r = sort_negative(negative_lc_2023, all_positive_Ids)
negative_lc_2024_g, negative_lc_2024_r = sort_negative(negative_lc_2024, all_positive_Ids)

Splitting data by filter: 100%|██████████| 9953/9953
Splitting data by filter: 100%|██████████| 13565/13565
Splitting data by filter: 100%|██████████| 7439/7439
Splitting data by filter: 100%|██████████| 18503/18503
Splitting data by filter: 100%|██████████| 15844/15844


In [7]:
# Grouping and saving to parquet:
all_negative_lc_g = pd.concat([negative_lc_2020_g, negative_lc_2021_g, negative_lc_2022_g, negative_lc_2023_g, negative_lc_2024_g]).reset_index(drop=True)
all_negative_lc_r = pd.concat([negative_lc_2020_r, negative_lc_2021_r, negative_lc_2022_r, negative_lc_2023_r, negative_lc_2024_r]).reset_index(drop=True)

all_negative_lc_g.to_parquet('../../data/negative/light_curves_g.parquet')
all_negative_lc_r.to_parquet('../../data/negative/light_curves_r.parquet')

## Extracting features

### Positive

In [8]:
positive_features_g, positive_features_r = extract_features(all_positive_lc_g), extract_features(all_positive_lc_r)
polars_features_g, polars_features_r = extract_features(polars_lc_g), extract_features(polars_lc_r)

Extracting features: 100%|██████████| 54/54
Extracting features: 100%|██████████| 57/57
Extracting features: 100%|██████████| 30/30
Extracting features: 100%|██████████| 34/34


In [9]:
# Saving to parquet:
positive_features_g.to_parquet('../../data/magcvs/features_g.parquet')
positive_features_r.to_parquet('../../data/magcvs/features_r.parquet')
polars_features_g.to_parquet('../../data/polars/features_g.parquet')
polars_features_r.to_parquet('../../data/polars/features_r.parquet')

### Negative

In [10]:
all_negative_features_g, all_negative_features_r = extract_features(all_negative_lc_g), extract_features(all_negative_lc_r)

Extracting features: 100%|██████████| 60006/60006
Extracting features: 100%|██████████| 65150/65150


In [11]:
# Saving to parquet:
all_negative_features_g.to_parquet('../../data/negative/features_g.parquet')
all_negative_features_r.to_parquet('../../data/negative/features_r.parquet')